# Kafi Streams: Complex Stream Processing Made Simple

### <i>Ralph Debusmann (Migros-Genossenschafts-Bund)</i>

<img src="../../pics/kafi_streams.jpg" style="width: 40%; height: 40%"/>

## Standing on the Shoulders of Giants

First off, Kafi Streams could not exist if it wasn't for the work of some real geniuses.

Matthias J. Sax (Confluent) is *the* representative for all the developers of *Kafka Streams*:

<img src="matthias.jpg" style="width: 30%; height: 30%"/>

Frank McSherry (Materialize) started all this in the 2010s at Microsoft Research (*Differential Dataflow*):

<img src="frank.jpg" style="width: 30%; height: 30%"/>

Mihai Budiu (with Leonid Ryzhyk, Feldera) created a cleaner and simpler version of Differential Dataflow - *DataBase Streaming Processor* (*DBSP*) - at VMware research:

<img src="mihai.jpg" style="width: 30%; height: 30%"/>

Bruno Rucy build a brilliant Open Source Python implementation of DBSP (*pydbsp*):

<img src="bruno.webp" style="width: 30%; height: 30%"/>


## My Kafka Story

I myself started studying AI in 1995, PhD in 2006...

<img src="xdg.png" style="width: 30%; height: 30%"/>

AI jobs in 2006? Not really. So - went to mundane software development in 2008...

<img src="sap.png" style="width: 30%; height: 30%"/>

6 years in (2014), found out about Scala, distributed systems, Spark, and... Kafka...

<img src="kafka.png" style="width: 20%; height: 20%"/>

The heureka moment: Kafka Streams (2016)... at the time, I thought stream processing would be omnipresent in a few years time...

<img src="kafka_streams.jpg" style="width: 40%; height: 40%"/>

But then - stream processing and Kafka Streams did get a lot of traction, but stream processing didn't become "omnipresent". It stayed in a niche. Why? I took up research again in 2023, leading up to co-authoring a book with Hubert Dulay:

<img src="streaming_databases.jpg" style="width: 30%; height: 30%"/>

At the end of writing this, I began being able to pinpoint why stream processing has never been able to really take off, writing a blog post which kind of went viral at Confluent:

<img src="mainstream.png" style="width: 30%; height: 30%"/>

So what is it that has held off stream processing? I think it's this...

## Leaky Abstractions in Stream Processing

### Imagine...

Imagine you would wish to build a stream processing pipeline.

Maybe just for joining a few topics before you ingest them into your database or search engine.

Or because you really have a low-latency, near real-time use case and you haven't touched stream processing before.

Should be easy, no?

You start using one of the most popular stream processing tools, e.g. Kafka Streams or Flink.

You get it to work.

And then you want to go to production.

<img src="boom.jpg" style="width: 30%; height: 30%"/>

### Classical Stream Processing Is Not Just Data Engineering

Regardless of whether you use SQL (e.g. ksqlDB on top of Kafka Streams or FlinkSQL on top of Flink) or a lower-level API (Kafka Streams DSL, Flink DataStream API), bringing stream processing to production is still incredibly hard.

Data Engineering skills are not enough. You need to be an expert in infrastructure and distributed systems as well.

Here are some of the concepts that you have to understand in order to build a stream processing pipeline in production:

Kafka Streams:
* co-partitioning
* keying
* KStream vs. KTable vs. GlobalKTable
* join semantics (Florian Troßbach: https://www.confluent.io/blog/crossing-streams-joins-apache-kafka/)
* suppress
* grace period

Flink:
* watermarks
* mini-batch
* multi-join
* allowed lateness
* side outputs for late data
* idleness detection
* checkpoint barriers
* barrier alignment/backpressure
* broadcast state/joins
* keyed stream transformation
* savepoints vs checkpoints
* changelog mode tuning
* sink tuning (append-only/upsert)

And, of course, a couple of concepts for both:

* RocksDB tuning
* eventual consistency
* exactly-once semantics
* non-determinism


### So What Happens?

(Too) often, stream processing projects are either stopped or not started at all. Just too hard. Just too expensive.

<img src="stop.jpg" style="width: 20%; height: 20%"/>

We need a world formula.


## DataBase Stream Processing (DBSP) - A Way Out? Can't It Be Easier?

During the research Hubert Dulay and me did for the O'Reilly book, we also talked with those behind Materialize and Feldera.

Materialize is based on Differential Dataflow (DD), a theory dating back to 2013 (Frank McSherry et al.):

<img src="dd.png" style="width: 40%; height: 40%"/>

Feldera is based on DataBase Stream Processing (DBSP), a simplified theory inspired by Differential Dataflow (Mihai Budiu et al. 2022):

<img src="dbsp.png" style="width: 40%; height: 40%"/>

Materialize and Feldera take a completely different approach as "classical" stream processing.

Not ad-hoc.

Not based on append-only streams. 

Not based on "infinite lists". 

But on *sets* and *relational algebra*.


## From leaky abstractions to relational algebra

We focus on DBSP going forward.

### What Is a Stream?

* "Classical" stream processing treats streams as just infinite streams of data.

* DBSP gives streams a *meaning*. For DBSP, streams are *sets* of *deltas*

### Z-Sets

* DBSP models deltas as *Z-Sets*

* Sounds fancy, but is just a function from anything (strings, tuples, JSONs...) to a *weight*

* E.g. this is a Z-Set: `{"A": 1, "B": 2}`

* What does it mean? The delta contains one record A (weight = 1) and two records B (weight = 2)

* Deletions = negative weights. E.g. this Z-Set deletes one of the Bs: `{"B": -1}`

* In stream processing, you get a continuous stream of deltas = Z-Sets.

* Z-Sets are combined by just summing up their weights. This is called *integration*:
  1. Input: `{"A": 1, "B": 2}` Output: `{"A": 1, "B": 2}`
  2. Input `{"B": -1}` Output: `{"A": 1, "B": 1}`
  3. Input `{"B": -1}` Output: `{"A": 1, "B": 0}`

* If weight = 0, you can remove the element from the set (e.g. for garbage collection). So: `{"A": 1, "B": 0}` = `{"A": 1}`

### Looks Innocent, But Could Be the World Formula for Streaming

Interestingly, if you build a stream processor on DBSP, you automatically get rid of many many of the complicated concepts that have plagued the adoption of stream processing so far. E.g.:

Kafka Streams:
* ~~co-partitioning~~ - partitions do not play any role any longer
* ~~keying~~ - you can e.g. use any field of your Kafka message for joining, even the header or the timestamp ;-)
* ~~KStream vs. KTable vs. GlobalKTable~~ - there is just one concept: the stream of deltas
* ~~join semantics~~ (Florian Troßbach: https://www.confluent.io/blog/crossing-streams-joins-apache-kafka/) - like in a database
* ~~suppress~~ 
* ~~grace period~~

Flink:
* watermarks (not quite, we need "waterlines" as in Feldera - next version ;-))
* ~~mini-batch~~ - any batch size from 1-N - same input, same output
* ~~multi-join~~
* ~~allowed lateness~~
* ~~side outputs for late data~~
* ~~idleness detection~~
* ~~checkpoint barriers~~
* ~~barrier alignment/backpressure~~
* ~~broadcast state/joins~~
* ~~keyed stream transformation~~
* ~~savepoints vs checkpoints~~
* ~~changelog mode tuning~~
* ~~sink tuning (append-only/upsert)~~


And, of course, a couple of concepts for both:

* RocksDB tuning - well, no spill-to-disk implemented yet, so that might still come in in some disguise ;-) 
* ~~eventual consistency~~ - strong consistency instead
* ~~exactly-once semantics~~ - we work with sets - so de-duplication is built-in
* ~~non-determinism~~ - deterministic - same input, same output, always


## Kafi Streams

Feldera (Mihai Budiu et al.) is a company around DBSP, based on an Open Source Rust implementation (https://github.com/feldera/feldera)

Bruno Rucy (for his PhD thesis) built a Python implementation of DBSP - *pydbsp* (https://github.com/brurucy/pydbsp) 

*Kafi Streams* is a Python library built on top of pydbsp by Bruno Rucy (https://github.com/xdgrulez/kafi)

Havily inspired and borrowing from Kafka Streams.

Let's see it in action using the classic word count example.

In [1]:
import sys

sys.path.insert(1, "../..")

import kafi.streams.topologynode
from kafi.streams.topologynode import TopologyNode as Tn

source_str = "lines"

source_tn = Tn.source(source_str)

root_tn = (
    source_tn
    .flatmap(lambda x: [word_str for word_str in x.split()])
    ._integrate()
    ._map(lambda x, w: ({"word": x, "count": w}, 1))
)

print(root_tn.mermaid())


```mermaid
graph TD
a49db7b0-c216-4863-b7bf-8c0163333d7d[source_lines] --> 6d77156c-23fa-4ff0-a782-09898b5b8083[flatmap_op]
c8b5c03a-063a-4940-a7e5-ae4a986fb038[_integrate_op] --> 9e9994ef-b55e-4aca-bd0d-8e598829714f[_map_op]
6d77156c-23fa-4ff0-a782-09898b5b8083[flatmap_op] --> c8b5c03a-063a-4940-a7e5-ae4a986fb038[_integrate_op]
```


In [2]:
import sys

sys.path.insert(1, "../..")

import kafi.streams.topologynode
from kafi.streams.topologynode import TopologyNode as Tn

source_str = "lines"

source_tn = Tn.source(source_str)
source_tn._to_zSet_function = source_tn._from_records

root_tn = (
    source_tn
    .flatmap(lambda x: [word_str for word_str in x.split()])
    ._integrate()
    ._map(lambda x, w: ({"word": x, "count": w}, 1))
)

root_tn.build()
root_tn._from_zSet_function = root_tn._to_records

#

line_str = "hello hello kafi streams"

print("Step 1:")
root_tn.push(source_str, [(line_str, 1)])
root_tn.latest()



Exception: Cannot build multiple non-sink nodes.

In [10]:
print("Step 2:")
root_tn.push(source_str, [(line_str, 1)])
root_tn.latest()

Step 2:


[({'word': 'hello', 'count': 4}, 1),
 ({'word': 'kafi', 'count': 2}, 1),
 ({'word': 'streams', 'count': 2}, 1)]

In [11]:
print("Step 3:")
root_tn.push(source_str, [(line_str, -1)])
root_tn.latest()


Step 3:


[({'word': 'hello', 'count': 2}, 1),
 ({'word': 'kafi', 'count': 1}, 1),
 ({'word': 'streams', 'count': 1}, 1)]

## Internal Consistency in Streaming Systems

Now for a more involved example.

Already in Hubert's and my O'Reilly book, we had an entire chapter dedicated to a (in my opinion) monumental blog post by Jamie Brandon (ex-Materialize, https://www.scattered-thoughts.net/writing/internal-consistency-in-streaming-systems/):

<img src="consistency.png" style="width: 40%; height: 40%"/>

It showed how "classical" stream processors like Kafka Streams and Flink can easily be "tricked" by some simple lines of SQL.

The input is a stream of transactions moving money between 10 distinct bank accounts. The amount of money moved is always 1.

First we create two views for the credits and debits of each account:

```sql
CREATE VIEW credits AS
SELECT
    to_account AS account, 
    sum(amount) AS credits
FROM
    transactions
GROUP BY
    to_account;

CREATE VIEW debits AS
SELECT
    from_account AS account, 
    sum(amount) AS debits
FROM
    transactions
GROUP BY
    from_account;
```    

Second, we calculate their balances (= credits - debits):

```sql
CREATE VIEW balance AS
SELECT
    credits.account AS account, 
    credits.credits - debits.debits AS balance
FROM
    credits
INNER JOIN debits 
    ON credits.account = debits.account;
```

Since money is only being moved around, never created or destroyed, the sum of all the balances should always be 0. This is how we get the total sum:

```sql
CREATE VIEW total AS
SELECT
    sum(balance)
FROM
    balance;
```


## Let's Put Flink SQL On It

When you run this on a "classical" stream processor like Flink, you see the effect of "eventual consistency" in action.

While you should only ever get one result - the sum 0, you get a constant flurry of mostly incorrect resulting sums.

<img src="chaos.png" style="width: 50%; height: 50%"/>

Some results are 0, some aren't... and only when you stop the input stream - it does give you the correct result - *eventually*.


## Let's Put Kafi Streams on It

For simplicity, in the examples presented here we run the examples without Kafka but directly on the Kafi Streams "TopologyNode" class. You can do the same with Kafka sources and sinks when you try it out yourself :)

### Kafi Streams Topology

In [ ]:
import sys

sys.path.insert(1, "../..")

import kafi.streams.topologynode
from kafi.streams.topologynode import TopologyNode as Tn

#

transaction_source_str = "transactions"

transaction_source_tn = Tn.source(transaction_source_str)

transaction_tn = transaction_source_tn.map(
    lambda x: {
        "from_account": x["from_account"],
        "to_account": x["to_account"],
        "amount": x["amount"]
    }
)
#
credits_tn = transaction_tn.group_by_sum(
    lambda x: x["to_account"],
    lambda x: x["amount"],
    lambda x, y: {"account": x,
                    "credits": y}
)
#
debits_tn = transaction_tn.group_by_sum(
    lambda x: x["from_account"],
    lambda x: x["amount"],
    lambda x, y: {"account": x,
                    "debits": y}
)
#
balance_tn = credits_tn.join_equi(
    debits_tn,
    lambda l: l["account"],
    lambda r: r["account"],
    lambda l, r: {
        "account": l["account"],
        "balance": l["credits"] - r["debits"]
    }
)
#
root_tn = balance_tn.sum(
    lambda x: x["balance"],
    lambda x: {"total": x}
)
#
root_tn.build()


### Throw Some Messages at It...

In [7]:
import random

def gen(n_int):
    record_dict_list = []
    for _ in range(n_int):
        record_dict = {"from_account": random.randint(0, 9),
                        "to_account": random.randint(0, 9),
                        "amount": 1}
        #
        record_dict_list.append(record_dict)
    #
    return record_dict_list

root_tn.push([(transaction_source_str, gen(10000))])
root_tn.latest()



[{'total': 0}]

### What Happened Here?

The first total was `[{'total': 0}]`

When we pushed more and more messages, it stayed that way... 

That is what we actually wanted. *Strong consistency*.

And what you cannot get in a classical *eventually consistent* stream processor without tuning.

### Garbage Collection

You might ask how Kafi Streams/pydbsp did that.

Doesn't it have to hold all the messages in memory to achieve strong consistency?

I mean we've been told for decades that strong consistency just cannot be achieved on unbounded streams...

See for yourself. We throw a lot more messages on Kafi Streams and check its state size...


...and you see the power of Z-Sets (and Bruno Rucy's ingenious pydbsp GC implementation) in action.

## Try the World Formula Yourself!

Kafi Streams is the first Kafka Streams-like library based on DBSP. Please try it out yourself to get a taste. IMHO, DBSP is incredible and could really be something like a world formula for stream processing.

Kafi Streams is not yet released on PyPI, but you can already play with it on GitHub: https://github.com/xdgrulez/kafi

### What's in It Already?

It offers a lot more than you could see in this talk (https://github.com/xdgrulez/kafi/tree/main/presentations/2026-06-09-Berlin_Buzzwords):
* all the features of Kafi (actually, Kafi Streams is just an extension of Kafi) presented e.g. at Current 2025 Bangalore
* e.g. support for Schema Registry, REST Proxy, "emulated Kafka" on disk, S3, Azure Blob Storage, "MirrorMaker as a one-liner" etc.
* support for Kafka sources and sinks
* support for fault-tolerant checkpointing to Kafka, S3 or Azure Blob Storage for fault tolerance and recovery
* support for Debezium sources and sinks

### What's Not Yet?

Kafi Streams is still kind of a hobby project with just me behind it. Some pieces are still missing:
* time window support (Feldera already has it, so doable) 
* spill state to disk/blob storage (pydbsp already has a configurable storage back-end, so doable)
* multicore support (pydbsp already has it, so doable)

And, of course:
* a lot of polishing, bug fixing and optimization in all places
* a first release
* good handwritten documentation




## Thank You for Having Been Here!

<img src="../../pics/kafi_streams.jpg" style="width: 30%; height: 30%"/>

In [ ]:
import importlib
importlib.reload(kafi.streams.topologynode)
print(sys.path)